# LLM-augmented BO: phase and amplitude

**AI prepares a prior and a starting point → GNLSE measures D_IO → AI chooses the next point → repeat.**

Select **GNLSE (Python)**. Edit the settings below, then run the cells in order.
The function details are in [the Python helper](../../../notebook_helpers/llm_bo/notebook.py).

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))
from llm_bo.notebook import run_experiment, show_results
from llm_bo.comparison import compare_to_previous

## 1. Settings

Use **100** or **81.943 kW**, with three adjustable launch amplitudes and two relative phases:

\[
a_m=\frac{c_m e^{i\phi_m}}{\sqrt{\sum_j c_j^2}},\qquad
0\le c_m\le100,\qquad \sum_m c_m=100,\qquad P_m/P=|a_m|^2,
\]
\[
\phi_1=0,\qquad \phi_2,\phi_3\in[-\pi,\pi)\quad\text{(radians, modulo }2\pi\text{)}.
\]

**Five stored variables describe four independent controls.** Python checks the
amplitude simplex and phase bounds, maps +π to −π, and ignores the phase of an absent mode.
The [reference setup](../Gradient%20descent/dio_phase_amplitude_optimization.ipynb) supplies the fiber,
solver settings, LP01 objective, and equal-amplitude, zero-phase baseline.
We minimize LP01 D_IO and require at least **50%** of baseline LP01 brightness.
The original finite-difference steps and local step caps are replaced by AI-selected points.

In [ ]:
POWER_KW = 100.0            # Choose 100.0 or 81.943.
MAX_TRIALS = 3             # Number of AI-selected test points.
MODEL = None  # Use the model configured in Codex.
THINKING_EFFORT = None  # Set an effort supported by your model to override.

## 2. Run the search

The short code excerpts show the main calls inside `run_experiment`. The helper
supplies the run folder, simulator, and baseline history. Run the cell after 2.4
to execute the full workflow.

### 2.1 Prepare AI

Use the selected **model and thinking effort**, and give AI the **context of this run,
optimization parameters, and ranges**. Let it set up a suitable prior model:

\[f(x)\sim\mathcal{GP}\!\left(m(x),k(x,x')\right).\]

AI can refine the GP's coordinates, kernel, and symmetry assumptions based on the
physical context. **Print its reasoning briefly:** the chosen prior and a short explanation.

`problem_card` packages the run context sent to AI. A shortened view:

```python
{
    "task": "Minimize LP01 D_IO",
    "power_kW": POWER_KW,
    "parameters": {
        **{c: {"bounds": [0, 100], "unit": "relative amplitude"}
           for c in ("c1", "c2", "c3")},
        **{phi: {"bounds": [-3.141592653589793, 3.141592653589793], "unit": "radian"}
           for phi in ("phi2_rad", "phi3_rad")},
    },
    "constraint": "c1+c2+c3=100; modal power fractions = c_i^2/sum(c_j^2)",
    "launch": "200 fs intensity-FWHM Gaussian; phi1=0; two variable relative phases",
    "fiber": "Silica step-index; length 2.86 m; carrier 1379.2 nm",
    "brightness": "LP01 photons >= 50% of equal-amplitude 1 mm baseline",
}
```

The full card also includes the exact D_IO definition, numerical settings,
and physical context such as the possible LP11 exchange symmetry. The periodic GP
embedding is `[w1, w2*cos(phi2), w2*sin(phi2), w3*cos(phi3), w3*sin(phi3)]`,
where `w` contains amplitude or power fractions chosen by AI. Any LP11 exchange
swaps both amplitude and phase together. Historical optima are withheld from AI.

```python
problem = problem_card(POWER_KW, phases=True)  # Context, parameters, and ranges.
ask = connect_llm(run_dir / "ai", max_calls=MAX_TRIALS + 1,
                  model=MODEL, reasoning_effort=THINKING_EFFORT)
plan, prior = prepare_ai(problem, ask)  # Prints the prior and brief explanation.
```

### 2.2 AI picks an initial test point in the range and starts initiating BO

AI chooses the starting amplitudes and relative phases along with the prior in the same call.
Python checks the amplitude and phase bounds and coefficient sum,
measures the equal-amplitude baseline for the brightness constraint, and starts the loop.

```python
point = check_point(plan["initial_point"], phases=True)  # Check AI's starting point.
EQUAL = problem["initial_reference"]  # Three equal amplitudes and two zero phases.
baseline_fine = simulator.measure(EQUAL, POWER_KW, CHECK_DZ)
reference_photons = baseline_fine["metrics"]["LP01_photons"]
```

`EQUAL` is the equal-amplitude input; `CHECK_DZ` is the 1 mm check step.

### 2.3 BO does one test point: run a GNLSE and get D_IO

Python runs GNLSE at the chosen input and gets one measured **D_IO**. It sends the
input amplitudes and phases, D_IO, brightness, validity, solver resolution, and measurement history
**back to Codex in a nice format** (structured JSON), asking **what to do next**.

```python
row = evaluate(point, simulator, POWER_KW, reference_photons, phases=True)
history.append(row)  # Save the measured input, D_IO, and diagnostics.
decision = choose_next(problem, history, calibration,
                       remaining=MAX_TRIALS - 1, ask=ask)
```

```python
# Inside choose_next (shortened):
decision = ask({
    "problem_card": problem,               # Context and bounds.
    "observations": history,               # Measured trials.
    "reference_measurements": calibration, # Baseline checks.
    "instruction": "Return exactly one new point or stop.",
}, NEXT_SCHEMA)  # Response: action, point, rationale.
```

The full JSON prompt also includes the best feasible result, an improvement target for D_IO, and the
remaining trial budget. Here, `MAX_TRIALS - 1` means one AI-selected trial has been used.

AI balances **exploring untested mixtures** with **improving the measured D_IO**,
while considering the brightness constraint. It returns a small dictionary:

```python
# Illustrative response; the proposed point still needs a GNLSE evaluation.
decision = {
    "action": "evaluate",
    "point": {"c1": 30.0, "c2": 35.0, "c3": 35.0,
              "phi2_rad": 1.5707963267948966, "phi3_rad": -1.0471975511965976},
    "rationale": "Test a mixture with nonzero relative phases and measure its D_IO and brightness.",
}
```

`action="evaluate"` supplies the next input in `point`; `action="stop"` uses
`point=None`. `rationale` briefly explains the choice and is printed. Python checks
that amplitudes sum to 100, all controls are in range, and the launch has not already
been tested, including equivalent phases at +π and −π. Distinct phases at the
same amplitudes remain distinct trial points.
With no trials left, AI must stop. The proposed point's D_IO and brightness
are determined by the next GNLSE evaluation.

### 2.4 AI does the acquisition function's job: Evaluate, update, repeat

**We can forget about the prior model now:** later choices use the measured data
directly; the GP is not updated. The **acquisition function is replaced by AI judgment**.
AI briefly explains its choice, then returns one new point or stops. If it chooses
to repeat, Python evaluates that point, updates the measurement history, and sends it back.

**Evaluate → update the data → AI chooses the next point or stops → repeat.**

With three trials, the limit is **six nonlinear solves and four AI calls**, including
baseline and final checks. Each run saves to a new folder.

In [ ]:
# File the saved campaign under its launch controls and search method.
from notebook_helpers.run_library import file_saved_run

run = run_experiment(
    phases=True,
    power_kW=POWER_KW,
    max_trials=MAX_TRIALS,
    model=MODEL,
    thinking_effort=THINKING_EFFORT,
)
file_saved_run(run.directory, "Modal amplitudes and relative phases", "LLM-assisted Bayesian optimization")

## 3. View the results

Show the measured history, brightness, final step-size check, pulse plots, amplitudes, power fractions, and relative phases.
This cell uses saved measurements and starts no new simulations.

In [ ]:
show_results(run)

The GP is used only for initialization. AI selects later points directly from the
measurements, using prompts adapted from [LLAMBO](https://arxiv.org/pdf/2402.03921).
Search uses **5 mm** steps; baseline and selected-point checks use **1 mm**, with
**Nt=2048** throughout. Reaching the trial limit does not establish convergence.

## Optional comparison with classical gradient descent

Run the matching phase-and-amplitude gradient-descent notebook at the same launch power and set the directory below to its printed run directory.

In [ ]:
CLASSICAL_RUN = None
if CLASSICAL_RUN is not None:
    comparison = compare_to_previous(run, previous_directory=CLASSICAL_RUN)